In [79]:
import os
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
path = os.path.join( "..", "data", "processed", "accidents_clean.csv")
accidents_clean = pd.read_csv(path)

In [80]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from itertools import combinations

def cramers_v_corrected(x, y):
    confusion_matrix = pd.crosstab(x, y)
    if confusion_matrix.shape[0] < 2 or confusion_matrix.shape[1] < 2:
        return np.nan, np.nan  # Trop peu de modalités

    chi2, p, _, _ = chi2_contingency(confusion_matrix, correction=False)
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape

    # Correction de biais (Bergsma)
    phi2corr = max(0, phi2 - ((k - 1)*(r - 1)) / (n - 1))
    rcorr = r - ((r - 1)**2) / (n - 1)
    kcorr = k - ((k - 1)**2) / (n - 1)

    v = np.sqrt(phi2corr / min((kcorr - 1), (rcorr - 1)))
    return v, p

def analyser_categorielle_v_cramer(df, cat_vars, target, seuil_redondance=0.4):
    print("\n=== 🔍 Association avec la target ===")
    cramer_target = {}
    for var in cat_vars:
        v, p = cramers_v_corrected(df[var], df[target])
        cramer_target[var] = v
        if not np.isnan(v):
            print(f"📌 {var} vs {target} → V de Cramér = {v:.3f}, p = {p:.3g}")

    print("\n=== 🔗 Détection de redondances entre variables catégorielles ===")
    redondantes = []
    for var1, var2 in combinations(cat_vars, 2):
        v, _ = cramers_v_corrected(df[var1], df[var2])
        if not np.isnan(v) and v > seuil_redondance:
            redondantes.append((var1, var2, v))
            print(f"⚠️  {var1} et {var2} sont corrélées (V = {v:.3f})")

    if not redondantes:
        print("✅ Aucune redondance forte détectée.")
    return cramer_target, redondantes

In [83]:
excluded = ['id', 'date', 'target','lat','long','age']
included = [col for col in accidents_clean.columns if col not in excluded]
target = 'grav'

analyser_categorielle_v_cramer(accidents_clean, included, target)


=== 🔍 Association avec la target ===
📌 obs vs grav → V de Cramér = 0.159, p = 0
📌 obsm vs grav → V de Cramér = 0.144, p = 0
📌 choc vs grav → V de Cramér = 0.122, p = 0
📌 manv vs grav → V de Cramér = 0.162, p = 0
📌 motor vs grav → V de Cramér = 0.101, p = 0
📌 place vs grav → V de Cramér = 0.146, p = 0
📌 catu vs grav → V de Cramér = 0.175, p = 0
📌 grav vs grav → V de Cramér = 1.000, p = 0
📌 sexe vs grav → V de Cramér = 0.092, p = 0
📌 trajet vs grav → V de Cramér = 0.109, p = 0
📌 jour vs grav → V de Cramér = 0.002, p = 0.304
📌 mois vs grav → V de Cramér = 0.020, p = 8.01e-146
📌 an vs grav → V de Cramér = 0.009, p = 8.11e-27
📌 hrmn vs grav → V de Cramér = 0.061, p = 0
📌 lum vs grav → V de Cramér = 0.068, p = 0
📌 dep vs grav → V de Cramér = 0.161, p = 0
📌 com vs grav → V de Cramér = 0.287, p = 0
📌 agg vs grav → V de Cramér = 0.168, p = 0
📌 int vs grav → V de Cramér = 0.054, p = 0
📌 atm vs grav → V de Cramér = 0.032, p = 0
📌 col vs grav → V de Cramér = 0.152, p = 0
📌 catr vs grav → V de Cra

({'obs': 0.15865328516603125,
  'obsm': 0.143541366733404,
  'choc': 0.12226458437355421,
  'manv': 0.16213592903764001,
  'motor': 0.10095747650154671,
  'place': 0.14568703207091618,
  'catu': 0.1745034361030931,
  'grav': 1.0,
  'sexe': 0.09188406587607985,
  'trajet': 0.1094772083190185,
  'jour': 0.001849019737426005,
  'mois': 0.020266895107355277,
  'an': 0.008752238344764981,
  'hrmn': 0.0612675056406813,
  'lum': 0.06802146476580662,
  'dep': 0.16090098772957123,
  'com': 0.2866904864895877,
  'agg': 0.16834081763589528,
  'int': 0.05420292454959398,
  'atm': 0.03213200191225489,
  'col': 0.1515776930913992,
  'catr': 0.10770378463123376,
  'circ': 0.08069109010431977,
  'prof': 0.04007144304710091,
  'plan': 0.07278034185154672,
  'surf': 0.027288584717377124,
  'infra': 0.026743641643171633,
  'situ': 0.13027037571794794,
  'vma': 0.13135340787151706,
  'equipements': 0.2744899590576227,
  'catv_regroup': 0.234337806287813},
 [('obsm', 'catu', 0.47811567106805025),
  ('place

In [84]:
# On enlève les variables qui ont peu de poids (V Cramer <0.1) et celles qui ne sont pas informatives 
# ou difficilement exploitables dans un modèle (lat, long)

accidents_clean = accidents_clean.drop(['jour','mois','an','hrmn','sexe',
                                        'lum','int','atm','circ','prof','plan',
                                        'surf','infra','lat', 'long'],axis=1)

In [85]:
# suppression des variables catégorielles redondantes

accidents_clean = accidents_clean.drop(['catu','vma', 'dep','agg','catr'],axis=1)

In [ ]:
# réencodage de la variable hrmn

#accidents_clean['hrmn'] = pd.to_datetime(accidents_clean['hrmn'], format='%H:%M', errors='coerce')

# on récupère les heures
#accidents_clean['heure'] = accidents_clean['hrmn'].dt.hour

# encodage cyclique sur les heures
#accidents_clean['heure_sin'] = np.sin(2 * np.pi * accidents_clean['heure'] / 24)
#accidents_clean['heure_cos'] = np.cos(2 * np.pi * accidents_clean['heure'] / 24)


# on supprime des variables intermédiaires et inutiles (jour, mois et an corrèlent peu avec grav -> on supprime date)
#accidents_clean = accidents_clean.drop(['date','heure', 'hrmn'],axis=1)

In [86]:
# l'effet multiplicatif de age sur la gravité est proche de 1. Cette variable oa donc très peu de poids. 
# On peut sûrement la supprimer

accidents_clean = accidents_clean.drop(['age'],axis=1)

In [89]:
# Beaucoup de communes ont peu d'accidents.
# On regroupe les communes ayant moins de 10 accidents dans une catégorie "rare" pour
# éviter de surcharger le modèle avec des catégories peu informatives.
# Cela permet de réduire la complexité du modèle et d'améliorer la performance.

commune_counts = accidents_clean['com'].value_counts()
rare_communes = commune_counts[commune_counts < 10].index
mask = accidents_clean['com'].isin(rare_communes)
accidents_clean.loc[mask, 'com'] = 'rare'



In [90]:
accidents_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 618606 entries, 0 to 618605
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   obs           618606 non-null  float64
 1   obsm          618606 non-null  float64
 2   choc          618606 non-null  float64
 3   manv          618606 non-null  int64  
 4   motor         618606 non-null  int64  
 5   place         618606 non-null  float64
 6   grav          618606 non-null  int64  
 7   trajet        618606 non-null  float64
 8   com           618606 non-null  object 
 9   col           618606 non-null  float64
 10  situ          618606 non-null  float64
 11  equipements   618606 non-null  object 
 12  catv_regroup  618606 non-null  float64
dtypes: float64(8), int64(3), object(2)
memory usage: 61.4+ MB


In [91]:
for col in accidents_clean.columns:
    print(f"{col}: {accidents_clean[col].nunique()} valeurs uniques")
    print(accidents_clean[col].unique())
    print("-" * 40)
accidents_clean.nunique()

obs: 18 valeurs uniques
[ 0.  1.  4. 14.  9.  6. 15. 13.  8.  2. 16. 12.  3.  7. 17. 11.  5. 10.]
----------------------------------------
obsm: 7 valeurs uniques
[2. 0. 1. 9. 6. 4. 5.]
----------------------------------------
choc: 10 valeurs uniques
[5. 3. 1. 4. 2. 0. 8. 6. 7. 9.]
----------------------------------------
manv: 27 valeurs uniques
[23 11  0  2 21  1  9 26 15 17  4 12 16 19 13 14  3 10  5 24 18 20  7 22
 25  6  8]
----------------------------------------
motor: 7 valeurs uniques
[1 6 0 3 5 2 4]
----------------------------------------
place: 10 valeurs uniques
[ 2.  1. 10.  3.  4.  7.  9.  6.  8.  5.]
----------------------------------------
grav: 4 valeurs uniques
[2 1 3 4]
----------------------------------------
trajet: 7 valeurs uniques
[0. 5. 9. 1. 4. 2. 3.]
----------------------------------------
com: 7281 valeurs uniques
['93053' '93066' '92036' ... '45110' '52480' '85306']
----------------------------------------
col: 7 valeurs uniques
[2. 6. 4. 3. 5. 7. 1.]
--

obs               18
obsm               7
choc              10
manv              27
motor              7
place             10
grav               4
trajet             7
com             7281
col                7
situ               7
equipements       86
catv_regroup      10
dtype: int64

In [ ]:
# dep et manv ont beaucoup de catégories -> prévilégier le target encoding
# obs : 18 catégories, à voir si target ou one hot encoding
# les autres variables catégorielles : one hot encoding
# heure_sin et heure_cos ok

# après l'encoding, on devrait avoir une petite centaine de variables ce qui devrait être gérable compte tenu des 600000+ entrées 